**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Example Chapter 11: Optimal Active Portfolio
In this example we estimate the optimal active tilt towards the zero-cost portfolio that goes long decile 1 short-term reversal portfolio and short decile 10 short-term reversal portfolio from the Ken French Data Library.


### Imports and Setup

In [1]:
# Import packages
%pip install -q --upgrade farms
import farms as fm
import pandas as pd
import numpy as np

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option("display.max_rows", 20) # Force truncation if DataFrame has more than 20 rows

Note: you may need to restart the kernel to use updated packages.


### Load in Data Using `farms.load_ken_french_data()`
For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#load_ken_french_data).

`load_ken_french_data()`

**Required Inputs**

- `data_type` — Selects the portfolio granularity. Potential values are `"deciles"` and `"quintiles"`. Example: `data_type='deciles'`.
- `strategy` — Selects the firm characteristic used to sort stocks. Example: `strategy='shorttermreversal'`.

**Optional Inputs**

- `include_factors` — Default is `None`. Adds factor returns to the portfolio DataFrame. Potential values are `None`, `"market"`, `"ff3"`, and `"ff5"`. Example: `include_factors='market'`.
- `frequency` — Default is `"monthly"`. Selects the observation frequency when it is available for the strategy. Potential values are `"daily"`, `"weekly"`, and `"monthly"`. Example: `frequency='monthly'`.

**Outputs**

- `DataFrame` — Portfolio returns indexed by observation date, with decimal return columns and the requested market-factor columns. Example: `df=fm.load_ken_french_data(...)`.

In [2]:
# Get monthly short-term reversal deciles and market factors
df = fm.load_ken_french_data(
    data_type='deciles',
    strategy='shorttermreversal',
    include_factors='market',
)
print(df)

           dec1    dec2    dec3    dec4    dec5    dec6    dec7    dec8    dec9   dec10  mkt-rf      rf
date                                                                                                   
1926-07 -0.0241  0.0204  0.0248  0.0098  0.0334  0.0204  0.0204  0.0312  0.0163  0.1049  0.0289  0.0022
1926-08  0.0133  0.0305  0.0438  0.0252  0.0444  0.0274  0.0354  0.0164  0.0082  0.0417  0.0264  0.0025
1926-09 -0.0450  0.0158 -0.0064  0.0078  0.0019  0.0069  0.0044 -0.0168  0.0546 -0.0238  0.0038  0.0023
1926-10 -0.1104 -0.0440 -0.0249 -0.0300 -0.0186 -0.0232 -0.0330 -0.0166 -0.0239 -0.0570 -0.0327  0.0032
1926-11 -0.0125  0.0642  0.0347  0.0389  0.0474  0.0298  0.0215  0.0193  0.0167  0.0151  0.0254  0.0031
...         ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...
2026-04  0.1662  0.0509  0.0424  0.1864  0.0702  0.1201  0.0881  0.1279  0.1065  0.0112  0.0995  0.0029
2026-05  0.0051  0.0000 -0.0014  0.0567 -0.0064  0.0455  0.0770 

### Define Inputs
In the code cell below we define the covariance matrix and the vector of expected returns.  Here you can enter your own covriance matrix and vector of expected returns as desired, with any number of securities. You can also load these inputs from your account on Google Drive, or your own GitHub repo.   

In [3]:
df = df.copy()  # avoid SettingWithCopyWarning

# Define the zero-cost portfolio
df['dec1_dec10'] = df['dec1'] - df['dec10']

# Estimate Expected excess returns
expected_returns = df[['mkt-rf', 'dec1_dec10']].mean().to_numpy()

# Estimate the covariance matrix
covariance_matrix = df[['mkt-rf', 'dec1_dec10']].cov().to_numpy()
print(expected_returns)
print(covariance_matrix)
# Note that the covariance matrix here is defined using the excess market return.
# Subtracting off the risk-free rate, which is close to constant, has very little
# effect on the covariance matrix.  In some applications is is common to define
# the covariance matrix using the excess market return.

[0.00695624 0.00801747]
[[0.00281067 0.00064891]
 [0.00064891 0.00393059]]


### Define the Optimal Active Portfolio
For full parameter details, see the [`farms` portfolio API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#factor_tilt_portfolio).

`factor_tilt_portfolio()`

This function fixes the first input series as the benchmark and optimizes the zero-cost factor tilt.

**Required Inputs**

- `expected_returns` — Expected returns ordered as the benchmark followed by zero-cost factor portfolios. Example: `expected_returns=expected_returns`.
- `covariance_matrix` — Covariance matrix in the same order as `expected_returns`. Example: `covariance_matrix=covariance_matrix`.

**Optional Inputs**

- `rf` — Default is `0.0`. Risk-free return used in the Sharpe-ratio objective. Example: `rf=0.0`.
- `base_index` — Default is `0`. Identifies the fixed benchmark exposure. Example: `base_index=0`.

**Outputs**

- `tuple` — Returns `(exposures, expected_return, volatility)`. The first exposure is `1.0`; the remaining exposures are optimized factor tilts. Example: `active_exposures, active_return, active_volatility = fm.factor_tilt_portfolio(...)`.

In [4]:
active_exposures, active_return, active_volatility = fm.factor_tilt_portfolio(
    expected_returns,
    covariance_matrix,
    rf=0.0,
    base_index=0,
)

print(active_exposures)
print(active_return)
print(active_volatility)

[1.         0.81388399]
0.013481530821656828
0.08044002888575523


### Test Trading Strategy

In [5]:
# For comparison with Ex06-Ken_French.ipynb

# Create realized excess portfolio returns
excess = df[['mkt-rf', 'dec1_dec10']].to_numpy()   # shape (T,2)

# Calculate the excess returns as a 1D array
# @ performs an inner product
excess_array = excess @ active_exposures

# Convert the Series to a DataFrame
excess = pd.DataFrame(excess_array, index=df.index, columns=['excess'])

# compute total return
total_return = excess['excess'] + df['rf']

# Use the same summary function as Ex06 for a transparent comparison.
# With risk_free supplied, summary_stats uses excess-return volatility
# as the Sharpe-ratio denominator.
active_summary = fm.summary_stats(
    returns=total_return.to_frame(name='active_strategy'),
    risk_free=df['rf'],
    frequency='monthly',
)
active_summary

Average Return: 0.194
Annualized Volatility: 0.278
Annualized Sharpe Ratio: 0.582
